In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from pypdf import PdfReader
from collections import Counter
from mineru import MinerU
import os
from dotenv import load_dotenv
load_dotenv()

# 1.创建客户端
client = MinerU(token = os.getenv("MINERU_TOKEN"))

def prepare_docs():
    # 解析文件
    result = client.flash_extract("D:\破烂都放里\data\练习文档.pdf")
    md_text = result.markdown
    
    #切分
    headers_to_split_on =[
        ('#',"h1"),
        ('##',"h2"),
        ('###',"h3")
    ]
    spliter = MarkdownHeaderTextSplitter(headers_to_split_on,strip_headers=False)
    docs = spliter.split_text(md_text)
    return docs
docs = prepare_docs()

<>:14: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:14: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\zhanx\AppData\Local\Temp\ipykernel_21348\1638519419.py:14: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  result = client.flash_extract("D:\破烂都放里\data\练习文档.pdf")


In [2]:
from pymilvus import MilvusClient

client = MilvusClient(uri="http://localhost:19530")
collection_name = "my_collection_1"

In [3]:
from pymilvus import Function, FunctionType, DataType
def create_collection():
    #schema
    schema = client.create_schema(
    auto_id=False,
    enable_dynamic_field=False
   )
   #定义主键
    schema.add_field(field_name="id",datatype=DataType.INT64,is_primary=True)
    #定义稀疏向量
    schema.add_field(field_name="dense",datatype=DataType.FLOAT_VECTOR,dim=1024)
   #定义稀疏向量
    schema.add_field(field_name="sparse",datatype=DataType.SPARSE_FLOAT_VECTOR)
   #普通元数据，用于数据过滤
    schema.add_field(field_name="h2",datatype=DataType.VARCHAR,max_length=128)
   #原文档转向量
    schema.add_field(
    field_name="content",
    datatype=DataType.VARCHAR,
    max_length=65535,
    enable_analyzer=True,        #允许对该字段分词
    analyzer_params={"type":"chinese"}
   )
    bm25_function=Function(
        name="text_bm25_emb",
        function_type=FunctionType.BM25,
        input_field_names= ["content"], 
        output_field_names=["sparse"]
   )
    schema.add_function(bm25_function)
    text_embedding_function=Function(
        name="ali",
        function_type=FunctionType.TEXTEMBEDDING,
        input_field_names= ["content"], 
        output_field_names=["dense"],
        params={
            "provider":"dashscope",
            "model_name":"text-embedding-v3",
            "dim":1024
        }
    )
    schema.add_function(text_embedding_function)
    #添加索引
    #稠密向量索引
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="dense",
        index_type="AUTOINDEX",
        metric_type="COSINE"
    )
    #稀疏向量索引
    index_params.add_index(
        field_name="sparse",
        index_type="SPARSE_INVERTED_INDEX",
        metric_type="BM25"
    )
    #普通字段索引
    index_params.add_index(
        field_name="h2",
        index_type="AUTOINDEX"
    )
    #创建collection
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)
    client.create_collection(
        schema=schema,
        collection_name=collection_name,
        index_params=index_params
    )

create_collection()

In [4]:
#milvus接受dict,数据转换
data =[
    {
        "id": i+1,
        "content":doc.page_content,
        "h2":doc.metadata.get("h2","")
    }
    for i,doc in enumerate(docs)     #遍历同时拿到序号和对象
]
#分批插入
batch_size=20
for start in range(0,len(data),batch_size):
    batch = data[start:start + batch_size]
    res = client.insert(collection_name,batch)

In [9]:
from pymilvus import AnnSearchRequest, RRFRanker

def hybrid_search(query:str,ranker):
    #稠密
    dense_request=AnnSearchRequest(
        data=[query],
        anns_field="dense" ,         #检索字段名
        limit=3,
        param={}
    )
    #稀疏
    sparse_request=AnnSearchRequest(
        data=[query],
        anns_field="sparse" ,         #检索字段名
        limit=3,
        param={}
    )
    #混合
    reps = [dense_request,sparse_request]
    res = client.hybrid_search(
        collection_name=collection_name,
        reqs=reps,
        ranker=ranker,
        limit=2,
        output_fields=["content","h2"]
    )
    for hits in res:
        print("TopK result:")
        for hit in hits:
            print(hit)

    return res[0]

In [ ]:
# from openai import OpenAI

# base_url = os.getenv("DASHSCOPE_BASE_URL")
# api_key = os.getenv("DASHSCOPE_API_KEY")
# # 模型
# llm = OpenAI(
#     api_key=api_key,
#     base_url=ba
# )

# def ask(question):
#     #混合检索
#     ranker = RRFRanker(k=60)
#     hits = hybrid_search(question,ranker)
#     #拼上下文
#     context = ""
#     for h in hits:
#         chapter = h["entity"].get("h2","未知章节")
#         content = h["entity"]["content"]
#         piece = f"【来源：{chapter}】\n{content}"
#         context = context + piece +"\n\n"
#     context = context.strip()
#     if not context:
#         return "检索没有命中任何内容，请确认数据是否已插入。"
#     #prompt
#     prompt = f"""你是文献问答助手 PaperPilot。请根据下面的资料回答问题。

# 要求：
# 1. 只能使用下面提供的资料回答，禁止编造资料里没有的内容；
# 2. 回答中注明信息来源的章节；
# 3. 资料里没有足够信息时，直接回答"资料中没有找到相关内容"。

# 资料：
# {context}

# 问题：{question}
# """
#     response = llm.chat.completions.create(
#         model="qwen-plus",
#         messages=[{"role":"user","content":prompt}],
#         temperature=0.1
#     )

#     return response.choices[0].message.content.strip()

# # ===== 测试 =====
# print(ask("Milvus 的稀疏向量索引用什么类型？"))
# print("\n" + "=" * 60 + "\n")
# print(ask("RRF 融合策略里 k 参数的作用是什么？"))
# print("\n" + "=" * 60 + "\n")
# print(ask("这篇文档的作者获过什么奖？"))

In [12]:
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
# ================Agentic RAG===================

# 1.加载环境变量
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

# 2.初始化模型
model = init_chat_model(
    "qwen3.7-plus",
    model_provider="openai", 
    base_url=base_url,
    api_key=api_key)

#定义工具
@tool
def retrieve_docs(question: str):
    """检索知识库，获取与用户提问相关的知识片段"""
    
    ranker = RRFRanker(k=60)
    hits = hybrid_search(question,ranker)
    #拼上下文
    context = ""
    for h in hits:
        chapter = h["entity"].get("h2","未知章节")
        content = h["entity"]["content"]
        piece = f"【来源：{chapter}】\n{content}"
        context = context + piece +"\n\n"
    context = context.strip()

    return context

# 4.定义Agent，绑定model和Tool
agent = create_agent(
    model,
    tools=[retrieve_docs],
    system_prompt="你是一个文献问答助手，回答与文献相关问题时，必须调用工具检索文档，根据文档回答用户的问题，不能自己编造答案。答案要简短，直接引用原文。"
)


In [13]:
from langchain_core.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage("Milvus 的稀疏向量索引用什么类型？")]} # 查询重写
)
for message in response['messages']:
    message.pretty_print()

TopK result:
{'id': 10, 'distance': 0.032786883413791656, 'entity': {'content': '## 3.3 向量数据库的索引与检索机制  \n向量数据库负责存储切分块的向量表示并支持高效的近似最近邻搜索。本文选用 Milvus 作为向量数据库，其支持多种索引类型，其中 HNSW（Hierarchical Navigable Small World）索引在搜索精度与延迟之间提供了良好的折中。  \nHNSW 是一种基于多层图结构的近似最近邻搜索算法。其底层包含全部数据点，上层则由从下层采样的子集构成，形成类似“跳表”的层次化导航结构。搜索从顶层入口点开始，贪婪地移动至当前层最近的邻居，达到局部最优后下降至下一层继续搜索，最终在底层完成精细化的最近邻查找。该结构使得搜索过程能够在保持较高召回率的同时显著降低计算复杂度。  \n在 Milvus 中构建 HNSW 索引时，关键参数包括 M 与 efConstruction。M 控制每个节点在每层图中的最大连接数，较高的 M 值提升召回率但增加内存开销，Milvus 默认值为 30，推荐范围为[5,100]。efConstruction 控制索引构建时考虑的候选邻居数量，值越高图质量越好但构建时间越长。本文实验中将 M设置为 16，efConstruction 设置为 200，在构建效率与索引质量之间取平衡。搜索阶段的 ef 参数设置为 64，以在 Top-3 检索场景下保证足够的候选探索范围。', 'h2': '3.3 向量数据库的索引与检索机制'}}
{'id': 5, 'distance': 0.016129031777381897, 'entity': {'content': '## 2.2 向量表示与语义检索  \n向量表示（Embedding）是指将文本映射为固定维度稠密向量的过程，其目标是使语义相近的文本在向量空间中具有较小的距离。在 RAG 系统中，向量化模型负责将文献切分块与用户查询分别编码为向量，随后通过近似最近邻搜索在向量数据库中找出与查询向量最接近的文献片段。  \n与传统的稀疏检索方法（如 BM25）相比，稠密向量检索能够捕捉词汇层面的语义相似性，而非仅依赖词项重叠。BM25 基于词频与逆文档频率计算查询与文档的相关性得分，对精确术